In [1]:
import numpy as np
import tensorflow as tf

In [7]:
with open('reviews.txt', 'r') as f:
    reviews = f.read()
with open('labels.txt', 'r') as f:
    labels = f.read()

In [8]:
reviews[:2000]


'bromwell high is a cartoon comedy . it ran at the same time as some other programs about school life  such as  teachers  . my   years in the teaching profession lead me to believe that bromwell high  s satire is much closer to reality than is  teachers  . the scramble to survive financially  the insightful students who can see right through their pathetic teachers  pomp  the pettiness of the whole situation  all remind me of the schools i knew and their students . when i saw the episode in which a student repeatedly tried to burn down the school  i immediately recalled . . . . . . . . . at . . . . . . . . . . high . a classic line inspector i  m here to sack one of your teachers . student welcome to bromwell high . i expect that many adults of my age think that bromwell high is far fetched . what a pity that it isn  t   \nstory of a man who has unnatural feelings for a pig . starts out with a opening scene that is a terrific example of absurd comedy . a formal orchestra audience is tu

In [9]:
from string import punctuation
all_text = ''.join([c for c in reviews if c not in punctuation])
reviews = all_text.split('\n')

all_text = ' '.join(reviews)
words = all_text.split()

In [10]:
all_text[:2000]


'bromwell high is a cartoon comedy  it ran at the same time as some other programs about school life  such as  teachers   my   years in the teaching profession lead me to believe that bromwell high  s satire is much closer to reality than is  teachers   the scramble to survive financially  the insightful students who can see right through their pathetic teachers  pomp  the pettiness of the whole situation  all remind me of the schools i knew and their students  when i saw the episode in which a student repeatedly tried to burn down the school  i immediately recalled          at           high  a classic line inspector i  m here to sack one of your teachers  student welcome to bromwell high  i expect that many adults of my age think that bromwell high is far fetched  what a pity that it isn  t    story of a man who has unnatural feelings for a pig  starts out with a opening scene that is a terrific example of absurd comedy  a formal orchestra audience is turned into an insane  violent m

In [11]:
words[:100]


['bromwell',
 'high',
 'is',
 'a',
 'cartoon',
 'comedy',
 'it',
 'ran',
 'at',
 'the',
 'same',
 'time',
 'as',
 'some',
 'other',
 'programs',
 'about',
 'school',
 'life',
 'such',
 'as',
 'teachers',
 'my',
 'years',
 'in',
 'the',
 'teaching',
 'profession',
 'lead',
 'me',
 'to',
 'believe',
 'that',
 'bromwell',
 'high',
 's',
 'satire',
 'is',
 'much',
 'closer',
 'to',
 'reality',
 'than',
 'is',
 'teachers',
 'the',
 'scramble',
 'to',
 'survive',
 'financially',
 'the',
 'insightful',
 'students',
 'who',
 'can',
 'see',
 'right',
 'through',
 'their',
 'pathetic',
 'teachers',
 'pomp',
 'the',
 'pettiness',
 'of',
 'the',
 'whole',
 'situation',
 'all',
 'remind',
 'me',
 'of',
 'the',
 'schools',
 'i',
 'knew',
 'and',
 'their',
 'students',
 'when',
 'i',
 'saw',
 'the',
 'episode',
 'in',
 'which',
 'a',
 'student',
 'repeatedly',
 'tried',
 'to',
 'burn',
 'down',
 'the',
 'school',
 'i',
 'immediately',
 'recalled',
 'at',
 'high']

In [12]:
from collections import Counter
counts = Counter(words)
vocab = sorted(counts, key=counts.get, reverse=True)
vocab_to_int = {word: ii for ii, word in enumerate(vocab, 1)}

reviews_ints = []
for each in reviews:
    reviews_ints.append([vocab_to_int[word] for word in each.split()])

In [13]:
labels = labels.split('\n')
labels = np.array([1 if each == 'positive' else 0 for each in labels])

In [14]:
review_lens = Counter([len(x) for x in reviews_ints])
print("Zero-length reviews: {}".format(review_lens[0]))
print("Maximum review length: {}".format(max(review_lens)))

Zero-length reviews: 0
Maximum review length: 2514


In [15]:
# Filter out that review with 0 length
reviews_ints = [each for each in reviews_ints if len(each) > 0]

In [16]:
seq_len = 200
features = np.zeros((len(reviews), seq_len), dtype=int)
for i, row in enumerate(reviews_ints):
    features[i, -len(row):] = np.array(row)[:seq_len]

In [17]:
features[:10,:100]


array([[    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 14966,   316,     6,
            3,  1078,   204,     8,  2163,    32,     1,   170,    57,
           15,    49,    76,  5441,    44,   421,   113,   139,    15,
         4018,    61,   149,     9,     1,  4999,  5778,   475,    70,
            5,   258,    12, 14966,   316,    13,  2288,     6,    73,
         2398],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     

In [18]:
split_frac = 0.8
split_idx = int(len(features)*0.8)
train_x, val_x = features[:split_idx], features[split_idx:]
train_y, val_y = labels[:split_idx], labels[split_idx:]

test_idx = int(len(val_x)*0.5)
val_x, test_x = val_x[:test_idx], val_x[test_idx:]
val_y, test_y = val_y[:test_idx], val_y[test_idx:]

print("\t\t\tFeature Shapes:")
print("Train set: \t\t{}".format(train_x.shape), 
      "\nValidation set: \t{}".format(val_x.shape),
      "\nTest set: \t\t{}".format(test_x.shape))

			Feature Shapes:
Train set: 		(10807, 200) 
Validation set: 	(1351, 200) 
Test set: 		(1351, 200)


In [19]:
lstm_size = 256
lstm_layers = 1
batch_size = 500
learning_rate = 0.001

In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [10]:
texts = [
    "i love machine learning",
    "tensorflow is powerful",
    "deep learning is amazing",
    "i love coding"
]


In [11]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)


In [12]:
vocab = tokenizer.word_index
n_words = len(vocab) + 1

In [14]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# example (make sure sequences and labels exist!)
sequences = [[1, 2, 3], [4, 5], [6]]
labels = [1, 0, 1]

max_len = max(len(seq) for seq in sequences)
X = pad_sequences(sequences, maxlen=max_len)

y = np.array(labels)

print(X)
print(y)

[[1 2 3]
 [0 4 5]
 [0 0 6]]
[1 0 1]


In [15]:
embed_size = 300

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_words, output_dim=embed_size, input_length=max_len),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


In [17]:
model.fit(X, y, epochs=5, batch_size=2)


Epoch 1/5


2/2 [==============================] - 11s 30ms/step - loss: 0.6892 - accuracy: 0.6667
Epoch 2/5
2/2 [==============================] - 0s 17ms/step - loss: 0.6731 - accuracy: 1.0000
Epoch 3/5
2/2 [==============================] - 0s 16ms/step - loss: 0.6757 - accuracy: 0.6667
Epoch 4/5
2/2 [==============================] - 0s 18ms/step - loss: 0.6546 - accuracy: 1.0000
Epoch 5/5
2/2 [==============================] - 0s 23ms/step - loss: 0.6491 - accuracy: 1.0000


In [21]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

n_words = len(vocab)

# Create the graph object
graph = tf.Graph()

# Add nodes to the graph
with graph.as_default():
    inputs_ = tf.placeholder(tf.int32, [None, None], name='inputs')
    labels_ = tf.placeholder(tf.int32, [None, None], name='labels')
    keep_prob = tf.placeholder(tf.float32, name='keep_prob')

Instructions for updating:
non-resource variables are not supported in the long term


In [24]:
# Size of the embedding vectors (number of units in the embedding layer)
embed_size = 300 

with graph.as_default():
    embedding = tf.Variable(tf.random_uniform((n_words, embed_size), -1, 1))
    embed = tf.nn.embedding_lookup(embedding, inputs_)

In [26]:
import tensorflow as tf

lstm_size = 128

# LSTM layer
lstm = tf.keras.layers.LSTM(lstm_size)

# Dropout
drop = tf.keras.layers.Dropout(0.5)

In [27]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_words, output_dim=300),
    tf.keras.layers.LSTM(lstm_size),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [31]:
lstm_size = 128
lstm_layers = 2   # 👈 REQUIRED
batch_size = 32

In [33]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

with graph.as_default():
    # Basic LSTM cell
    lstm = tf.compat.v1.nn.rnn_cell.BasicLSTMCell(lstm_size)
    
    # Add dropout
    drop = tf.compat.v1.nn.rnn_cell.DropoutWrapper(lstm, output_keep_prob=keep_prob)
    
    # Stack multiple layers
    cell = tf.compat.v1.nn.rnn_cell.MultiRNNCell([drop] * lstm_layers)
    
    # Initial state
    initial_state = cell.zero_state(batch_size, tf.float32)

C:\Users\Dell\AppData\Local\Temp\ipykernel_5324\3705668475.py:6: UserWarning: `tf.nn.rnn_cell.BasicLSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  lstm = tf.compat.v1.nn.rnn_cell.BasicLSTMCell(lstm_size)


In [2]:
import tensorflow as tf

In [3]:
import tensorflow as tf
tf.compat.v1.disable_eager_execution()

In [5]:
cell = tf.compat.v1.nn.rnn_cell.LSTMCell(num_units=64)

C:\Users\Dell\AppData\Local\Temp\ipykernel_4460\3227170923.py:1: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(num_units=64)


In [7]:
inputs_ = tf.compat.v1.placeholder(tf.float32, [None, 10, 50])

In [8]:
outputs, states = tf.compat.v1.nn.dynamic_rnn(
    cell,
    inputs_,
    dtype=tf.float32
)

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [10]:
graph = tf.Graph()

In [12]:
import tensorflow as tf

tf.compat.v1.disable_eager_execution()

graph = tf.Graph()

with graph.as_default():

    # 1. Input
    inputs_ = tf.compat.v1.placeholder(tf.float32, [None, 10, 50])

    # 2. RNN cell
    cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

    # 3. RNN
    outputs, states = tf.compat.v1.nn.dynamic_rnn(
        cell,
        inputs_,
        dtype=tf.float32
    )

    # 4. Dense layer (same graph ✅)
    predictions = tf.compat.v1.layers.dense(
        outputs[:, -1],
        1,
        activation=tf.nn.sigmoid
    )

C:\Users\Dell\AppData\Local\Temp\ipykernel_4460\728252271.py:13: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_4460\728252271.py:23: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  predictions = tf.compat.v1.layers.dense(


In [17]:
tf.compat.v1.reset_default_graph()

In [3]:
import tensorflow as tf

tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

# 1. Input
inputs_ = tf.compat.v1.placeholder(tf.int32, [None, None])

# 2. Embedding
n_words = 5000
embed_size = 128

embedding = tf.Variable(tf.random.uniform((n_words, embed_size), -1, 1))
embed = tf.nn.embedding_lookup(embedding, inputs_)

# 3. RNN cell   (this fixes your current error)
cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

# 4. Initial state
initial_state = cell.zero_state(tf.shape(inputs_)[0], tf.float32)

# 5. RNN
outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
    cell,
    embed,
    initial_state=initial_state
)


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


C:\Users\Dell\AppData\Local\Temp\ipykernel_8200\2118333930.py:17: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)


In [4]:
import tensorflow as tf

tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

# Create graph
graph = tf.Graph()

with graph.as_default():

    # Input
    inputs_ = tf.compat.v1.placeholder(tf.int32, [None, None])

    # Embedding
    embedding = tf.Variable(tf.random.uniform((5000, 128), -1, 1))
    embed = tf.nn.embedding_lookup(embedding, inputs_)

    # RNN cell
    cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

    # Initial state
    initial_state = cell.zero_state(tf.shape(inputs_)[0], tf.float32)

    # RNN (correct API)
    outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
        cell,
        embed,
        initial_state=initial_state
    )

C:\Users\Dell\AppData\Local\Temp\ipykernel_18736\962140344.py:19: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)


In [2]:
import tensorflow as tf

tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

# Create graph
graph = tf.Graph()

with graph.as_default():

    # Input
    inputs_ = tf.compat.v1.placeholder(tf.int32, [None, None])

    # Embedding
    embedding = tf.Variable(tf.random.uniform((5000, 128), -1, 1))
    embed = tf.nn.embedding_lookup(embedding, inputs_)

    # RNN cell
    cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

    # Initial state
    initial_state = cell.zero_state(tf.shape(inputs_)[0], tf.float32)

    # RNN (FIXED: use compat.v1)
    outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
        cell,
        embed,
        initial_state=initial_state
    )




Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


C:\Users\Dell\AppData\Local\Temp\ipykernel_3508\1170001157.py:19: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)


In [7]:
cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

C:\Users\Dell\AppData\Local\Temp\ipykernel_3508\1891950687.py:1: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)


In [10]:
import tensorflow as tf

tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

# 1) Inputs
inputs_ = tf.compat.v1.placeholder(tf.int32, [None, None])

# 2) Embedding
vocab_size = 5000
embed_size = 128
embedding = tf.Variable(tf.random.uniform((vocab_size, embed_size), -1, 1))
embed = tf.nn.embedding_lookup(embedding, inputs_)  # <-- defines 'embed'

# 3) RNN cell
cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)

# 4) Initial state
initial_state = cell.zero_state(tf.shape(inputs_)[0], tf.float32)

# 5) RNN
outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
    cell,
    embed,
    initial_state=initial_state
)

# 6) Output layer
predictions = tf.compat.v1.layers.dense(
    outputs[:, -1],
    1,
    activation=tf.nn.sigmoid
)

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


C:\Users\Dell\AppData\Local\Temp\ipykernel_3508\306191562.py:16: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_3508\306191562.py:29: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  predictions = tf.compat.v1.layers.dense(


In [11]:
outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
    cell,
    embed,
    initial_state=initial_state
)

In [2]:
import tensorflow as tf

# Reset & disable eager (TF1 style)
tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

# Create graph
graph = tf.Graph()

with graph.as_default():

    # Inputs
    inputs_ = tf.compat.v1.placeholder(tf.int32, [None, None])
    labels_ = tf.compat.v1.placeholder(tf.float32, [None, 1])
    learning_rate = 0.001

    # Embedding
    embedding = tf.Variable(tf.random.uniform((5000, 128), -1, 1))
    embed = tf.nn.embedding_lookup(embedding, inputs_)

    # RNN
    cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)
    initial_state = cell.zero_state(tf.shape(inputs_)[0], tf.float32)

    outputs, final_state = tf.compat.v1.nn.dynamic_rnn(
        cell,
        embed,
        initial_state=initial_state
    )

    
    predictions = tf.compat.v1.layers.dense(
        outputs[:, -1],
        1,
        activation=tf.nn.sigmoid
    )

    cost = tf.compat.v1.losses.mean_squared_error(labels_, predictions)

    optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cost)




Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


C:\Users\Dell\AppData\Local\Temp\ipykernel_25444\1443045563.py:22: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  cell = tf.compat.v1.nn.rnn_cell.LSTMCell(64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_25444\1443045563.py:32: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  predictions = tf.compat.v1.layers.dense(


In [4]:
import tensorflow as tf

tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()

graph = tf.Graph()

In [4]:
def get_batches(x, y, batch_size=100):
    
    n_batches = len(x)//batch_size
    x, y = x[:n_batches*batch_size], y[:n_batches*batch_size]
    for ii in range(0, len(x), batch_size):
        yield x[ii:ii+batch_size], y[ii:ii+batch_size]

In [ ]:
import tensorflow as tf
import numpy as np

tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()

epochs = 10

# IMPORTANT: Saver must be created AFTER graph is built
graph = tf.Graph()

with graph.as_default():

    saver = tf.compat.v1.train.Saver()

with tf.compat.v1.Session(graph=graph) as sess:

    sess.run(tf.compat.v1.global_variables_initializer())

    iteration = 1

    for e in range(epochs):
        state = sess.run(initial_state)

        for ii, (x, y) in enumerate(get_batches(train_x, train_y, batch_size), 1):

            feed = {
                inputs_: x,
                labels_: y[:, None],
                keep_prob: 0.5,
                initial_state: state
            }

            loss, state, _ = sess.run(
                [cost, final_state, optimizer],
                feed_dict=feed
            )

            if iteration % 5 == 0:
                print(
                    "Epoch: {}/{}".format(e + 1, epochs),
                    "Iteration: {}".format(iteration),
                    "Train loss: {:.3f}".format(loss)
                )

            # -------- VALIDATION --------
            if iteration % 25 == 0:

                val_acc = []
                val_state = sess.run(
                    cell.zero_state(batch_size, tf.float32)
                )

                for x, y in get_batches(val_x, val_y, batch_size):

                    feed = {
                        inputs_: x,
                        labels_: y[:, None],
                        keep_prob: 1.0,
                        initial_state: val_state
                    }

                    batch_acc, val_state = sess.run(
                        [accuracy, final_state],
                        feed_dict=feed
                    )

                    val_acc.append(batch_acc)

                print("Val acc: {:.3f}".format(np.mean(val_acc)))

            iteration += 1

    saver.save(sess, "checkpoints/sentiment.ckpt")